# INSTALL LIBRARIES

In [1]:
!pip install fastapi nest-asyncio uvicorn transformers torch omegaconf pyngrok -q

# CREATE CONFIG FILE

In [2]:
yaml_config = """
model_path: "distilbert-base-cased-distilled-squad"
data_path: "data.txt"
confidence_threshold: 10
"""
with open("config.yaml", "w", encoding="utf-8") as f:
    f.write(yaml_config)

# CREATE DATA

In [3]:
context_data = """
Vietnam is a country in Southeast Asia. The capital of Vietnam is Hanoi, while Ho Chi Minh City is the largest city.

Hanoi is known for its centuries-old architecture and rich culture with Southeast Asian, Chinese and French influences.

Ho Chi Minh City, formerly known as Saigon, is a major financial and business hub.

The Mekong River is one of the longest rivers in the world and flows through Vietnam.

Mount Fansipan is the highest mountain in Vietnam.

The Earth revolves around the Sun, and the Sun is a star at the center of the Solar System.

Asia is the largest continent in the world, followed by Africa.

The Pacific Ocean is the largest ocean on Earth.

The Nile River is the longest river in the world.

Mount Everest is the highest mountain in the world, located in the Himalayas.
"""

with open("data.txt", "w", encoding="utf-8") as f:
    f.write(context_data)

# GET STARTED WITH THE MODEL

In [4]:
from transformers import DistilBertTokenizer, DistilBertModel
import torch
tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-cased-distilled-squad')
model = DistilBertModel.from_pretrained('distilbert-base-cased-distilled-squad')

question, text = "Who was Jim Henson?", "Jim Henson was a nice puppet"

inputs = tokenizer(question, text, return_tensors="pt")
with torch.no_grad():
    outputs = model(**inputs)

print(outputs)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/473 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/261M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-cased-distilled-squad
Key               | Status     |  | 
------------------+------------+--+-
qa_outputs.bias   | UNEXPECTED |  | 
qa_outputs.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BaseModelOutput(last_hidden_state=tensor([[[ 1.1810, -0.4073,  0.9986,  ..., -0.7445,  0.0380, -0.5510],
         [ 1.6172, -0.6785,  1.6932,  ..., -0.8216, -0.2387, -0.6187],
         [ 2.0840, -0.5496,  1.3313,  ..., -0.7791,  0.1698, -0.3950],
         ...,
         [ 0.2879, -0.1813,  1.2631,  ..., -0.2022,  0.4699,  0.5535],
         [ 0.6069, -0.1943,  0.7584,  ..., -0.5106, -0.4027, -0.4910],
         [ 1.0183, -0.8215,  0.9088,  ..., -0.8094,  0.8372, -0.2027]]]), hidden_states=None, attentions=None)


# BUILD MODEL

In [5]:
import torch
from omegaconf import OmegaConf
from transformers import DistilBertTokenizer, DistilBertForQuestionAnswering


class QuestionAnsweringModel:
    def __init__(self, config_path):
        self.config = OmegaConf.load(config_path)

        self.tokenizer = DistilBertTokenizer.from_pretrained(self.config.model_path)
        self.model = DistilBertForQuestionAnswering.from_pretrained(self.config.model_path)

        with open(self.config.data_path, "r", encoding="utf-8") as f:
            self.context = f.read()

    def __call__(self, question):
        if not any(word.lower() in self.context.lower() for word in question.split()):
            return "I don't know"

        inputs = self.tokenizer(question, self.context, return_tensors="pt")

        with torch.no_grad():
            outputs = self.model(**inputs)

        start_logits = outputs.start_logits
        end_logits = outputs.end_logits

        start_index = start_logits.argmax()
        end_index = end_logits.argmax()

        answer_tokens = inputs["input_ids"][0][start_index : end_index + 1]
        answer = self.tokenizer.decode(answer_tokens).strip()

        confidence = (start_logits.max() + end_logits.max()).item()

        if (
            confidence < self.config.confidence_threshold
            or answer == ""
            or len(answer.split()) > 20
        ):
            return "I don't know"

        return answer

# INITIALIZE MODEL

In [6]:
qa_model = QuestionAnsweringModel("./config.yaml")

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

In [7]:
question = "What is the largest city in Vietnam?"

answer = qa_model(question)

print(answer)

Ho Chi Minh City


# INITIALIZE API

In [8]:
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel

app = FastAPI()

app.add_middleware(
    CORSMiddleware,
    allow_origins=['*'],
    allow_credentials=True,
    allow_methods=['*'],
    allow_headers=['*'],
)

class QuestionInput(BaseModel):
    question: str

# Endpoints
@app.get('/')
async def root():
    return {
        "system": "LAB 1: Question Answering API",
        "model": "distilbert-base-cased-distilled-squad",
        "description": "The Question Answering API retrieves answers from a predefined dataset based on user queries and returns them in JSON format"
    }

@app.get('/health')
async def health():
    return {"status": "ok"}

@app.post('/predict')
async def predict(data: QuestionInput):
    if not data.question.strip():
        raise HTTPException(status_code=400, detail="Question can not be empty")
    try:
        answer = qa_model(data.question)
        return {"question": data.question, "answer": answer}
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

# RUN SERVER

In [9]:
import threading
import uvicorn

def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8000)

thread = threading.Thread(target=run_server, daemon=True)
thread.start()

print("Server started on http://0.0.0.0:8000")

Server started on http://0.0.0.0:8000


# CALL LOCAL API

In [10]:
import requests

response = requests.post(
    "http://127.0.0.1:8000/predict",
    json={"question": "What is the capital of Vietnam?"}
)

print(response.json())

INFO:     127.0.0.1:47814 - "POST /predict HTTP/1.1" 200 OK
{'question': 'What is the capital of Vietnam?', 'answer': 'Hanoi'}


# CALL PUBLIC API

In [ ]:
# TODO: Chạy lệnh dưới để tunnel ra bên ngoài
# ssh -p 443 -R0:localhost:8000 qr@a.pinggy.io

# CHÚ Ý: SAU KHI ĐÃ CÓ LINK THÌ NHẤN PHẢI RỒI CHỌN COPY TRONG MENU, NẾU BẤM CTRL + C SẼ TẮT KẾT NỐI


In [11]:
import requests

BASE_URL = "http://mcfaw-34-125-155-180.run.pinggy-free.link"

API_URL = f"{BASE_URL}/predict"

payload = {
    "question": "What is the capital of Vietnam?"
}

response = requests.post(API_URL, json=payload, timeout=30)
print(response.json())

INFO:     34.125.155.180:0 - "POST /predict HTTP/1.1" 200 OK
{'question': 'What is the capital of Vietnam?', 'answer': 'Hanoi'}


# TEST API

In [13]:
import requests

BASE_URL = "http://mcfaw-34-125-155-180.run.pinggy-free.link"

PREDICT_URL = f"{BASE_URL}/predict"

def print_result(label, response, expect_status=None):
    status = response.status_code
    try:
        body = response.json()
    except Exception:
        body = response.text

    status_ok = (expect_status is None) or (status == expect_status)
    print(f"{body}")
    if not status_ok:
        print(f"Expected status: {expect_status}")
    print()


# NHÓM 1: Endpoint cơ bản
print("=" * 55)
print("Nhóm 1: Endpoint cơ bản")
print("=" * 55)

print_result(
    "GET /",
    requests.get(f"{BASE_URL}/"),
    expect_status=200
)

print_result(
    "GET /health",
    requests.get(f"{BASE_URL}/health"),
    expect_status=200
)

# NHÓM 2: Happy path — câu hỏi có trong data.txt
print("=" * 55)
print("Nhóm 2: Happy path (câu hỏi có trong data.txt)")
print("=" * 55)

print_result(
    "POST /predict",
    requests.post(PREDICT_URL, json={"question": "What is the capital of Vietnam?"}),
    expect_status=200
)

print_result(
    "POST /predict",
    requests.post(PREDICT_URL, json={"question": "What is the largest ocean?"}),
    expect_status=200
)

print_result(
    "POST /predict",
    requests.post(PREDICT_URL, json={"question": "What is the highest mountain in Vietnam?"}),
    expect_status=200
)

# NHÓM 3: Câu hỏi ngoài phạm vi data.txt
print("=" * 55)
print("Nhóm 3: Câu hỏi ngoài phạm vi data.txt")
print("=" * 55)

print_result(
    "POST /predict",
    requests.post(PREDICT_URL, json={"question": "Who is the president of Mars?"}),
    expect_status=200
)

print_result(
    "POST /predict",
    requests.post(PREDICT_URL, json={"question": "What is the population of France?"}),
    expect_status=200
)

# NHÓM 4: Validation lỗi đầu vào
print("=" * 55)
print("Nhóm 4: Validation lỗi đầu vào")
print("=" * 55)
print_result(
    "POST /predict",
    requests.post(PREDICT_URL, json={"question": ""}),
    expect_status=400
)

print_result(
    "POST /predict",
    requests.post(PREDICT_URL, json={"question": "   "}),
    expect_status=400
)

print_result(
    "POST /predict",
    requests.post(PREDICT_URL, json={}),
    expect_status=422
)

print_result(
    "POST /predict",
    requests.post(PREDICT_URL, data={"question": "What is the capital?"}),
    expect_status=422
)

print_result(
    "POST /predict",
    requests.post(PREDICT_URL, json={"question": 123}),
    expect_status=200
)

# NHÓM 5: Routing sai
print("=" * 55)
print("Nhóm 5: Endpoint không tồn tại")
print("=" * 55)

print_result(
    "GET /unknown",
    requests.get(f"{BASE_URL}/unknown"),
    expect_status=404
)

print_result(
    "GET /predict",
    requests.get(PREDICT_URL),
    expect_status=405
)

Nhóm 1: Endpoint cơ bản
INFO:     34.125.155.180:0 - "GET / HTTP/1.1" 200 OK
{'system': 'LAB 1: Question Answering API', 'model': 'distilbert-base-cased-distilled-squad', 'description': 'The Question Answering API retrieves answers from a predefined dataset based on user queries and returns them in JSON format'}

INFO:     34.125.155.180:0 - "GET /health HTTP/1.1" 200 OK
{'status': 'ok'}

Nhóm 2: Happy path (câu hỏi có trong data.txt)
INFO:     34.125.155.180:0 - "POST /predict HTTP/1.1" 200 OK
{'question': 'What is the capital of Vietnam?', 'answer': 'Hanoi'}

INFO:     34.125.155.180:0 - "POST /predict HTTP/1.1" 200 OK
{'question': 'What is the largest ocean?', 'answer': 'Pacific Ocean'}

INFO:     34.125.155.180:0 - "POST /predict HTTP/1.1" 200 OK
{'question': 'What is the highest mountain in Vietnam?', 'answer': 'Mount Everest'}

Nhóm 3: Câu hỏi ngoài phạm vi data.txt
INFO:     34.125.155.180:0 - "POST /predict HTTP/1.1" 200 OK
{'question': 'Who is the president of Mars?', 'answer'